# 01 · Data prep

| Setting | Value |
|---|---|
| Accelerator | **None (CPU)** is enough — only tokenization and pure-Python scope tracking |
| Internet | **ON** (pip install, Hugging Face dataset + tokenizer download) |
| Attach | the code (Kaggle dataset `scopesum-code`) unless you set `REPO_URL` |
| Expected runtime | ~10 min download + ~30–60 min for the default 10M-token budget |

Output: `/kaggle/working/scopesum_data` (windows, examples for modes `none`/`fixed`/`scope`, `stats.json` with variant B's `K`).
After it finishes: **Save Version**, then create a Kaggle dataset from this notebook's output (e.g. `scopesum-data`) and attach it to `02_train`.

In [ ]:
# --- get the code: git clone (needs Internet) OR an attached Kaggle dataset ---
REPO_URL = "https://github.com/loaferct/scopesum.git"  # private repo: needs the Kaggle Secret GITHUB_TOKEN; "" = use an attached code dataset
CODE_DATASET_DIR = "/kaggle/input"  # searched for a folder containing scopesum/__init__.py

import glob, os, shutil, subprocess, sys
CODE_DIR = "/kaggle/working/scopesum"

def clone_repo(url, dest, secret_name="GITHUB_TOKEN"):
    """git clone; uses a GitHub token from Kaggle Secrets for private repos (never printed)."""
    auth_url = url
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(secret_name)
        auth_url = url.replace("https://", f"https://x-access-token:{token}@")
    except Exception as e:
        print(f"No Kaggle Secret {secret_name!r} ({type(e).__name__}); trying an anonymous clone")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, dest], capture_output=True, text=True)
    if r.returncode != 0:
        print("git clone failed:", r.stderr.replace(auth_url, url)[-500:])
        return False
    subprocess.run(["git", "-C", dest, "remote", "set-url", "origin", url])  # drop the token from .git/config
    print("cloned", url, subprocess.run(["git", "-C", dest, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
    return True

if not os.path.exists(os.path.join(CODE_DIR, "scopesum", "__init__.py")):
    if not (REPO_URL and clone_repo(REPO_URL, CODE_DIR)):
        hits = [h for h in glob.glob(CODE_DATASET_DIR + "/**/scopesum/__init__.py", recursive=True) if "/tests/" not in h]
        assert hits, "Attach the scopesum code dataset or set REPO_URL"
        src = os.path.dirname(os.path.dirname(hits[0]))
        shutil.copytree(src, CODE_DIR)  # copy: /kaggle/input is read-only
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print("code dir:", CODE_DIR, os.listdir(CODE_DIR))

In [ ]:
!pip uninstall -y -q torchao  # Kaggle ships torchao 0.10; peft>=0.21 refuses it and we do not use torchao
!pip install -q -r requirements.txt
!python -m pytest -q tests -p no:cacheprovider

In [ ]:
from scopesum.config import load_config

DEBUG = False  # True = tiny run (a few hundred files) to check the pipeline end to end
overrides = {"data": {"token_budget": 300_000, "max_files": 400}} if DEBUG else {}
cfg = load_config("configs/base.yaml", overrides=overrides)
print(cfg.data)

## Discovery: what does the dataset actually contain?
Check that `text_field`, `repo_field`, `path_field` printed below exist in the row keys. If not, fix them in `configs/base.yaml` (or via `overrides`).

In [ ]:
from scopesum.data import discover_dataset
discover_dataset(cfg)

## Discovery: tokenizer, `<sum>` id, offset mapping, scope tracking on real tokens

In [ ]:
from scopesum.data import check_offsets, load_tokenizer, resolve_model_path, resolve_sum_token_id
from scopesum.scope_tracker import track_scopes

model_path = resolve_model_path(cfg)
tok = load_tokenizer(cfg, model_path)
sum_id = resolve_sum_token_id(tok, cfg)
print("tokenizer:", type(tok).__name__, "vocab:", len(tok))
print("bos/eos/pad:", tok.bos_token_id, tok.eos_token_id, tok.pad_token_id)
print("<sum> =", cfg.model.sum_token, "->", sum_id, repr(tok.convert_ids_to_tokens(sum_id)))

sample = '''import os\n\nclass A:\n    @property\n    def x(self):\n        """doc\n  odd indent"""\n        return 1\n\n    def y(self, a,\n          b):\n        return a + b\n\nprint(A().x)\n'''
print(check_offsets(tok, sample))
enc = tok(sample, add_special_tokens=False, return_offsets_mapping=True)
info = track_scopes(sample, enc["offset_mapping"])
for t, (s, e) in enumerate(enc["offset_mapping"]):
    sid = info.token_scope[t]
    name = info.scopes[sid].qualname if sid >= 0 else "?"
    print(f"{t:3d} {sample[s:e]!r:14} scope={name:10} header={info.token_is_header[t]}")
print("closures (insert_before, scope):", [(i, info.scopes[s].qualname) for i, s in info.closures])

## Build and save the dataset

In [ ]:
import json, time
from scopesum.data import prepare_dataset

t0 = time.time()
stats = prepare_dataset(cfg, tok, sum_id)
print(f"done in {(time.time() - t0) / 60:.1f} min")
print(json.dumps(stats, indent=2))
print("\n>>> Variant B chunk size K =", stats["chunk_size"],
      "| avg scope length =", round(stats["avg_scope_len"], 1),
      "| <sum>/example C =", round(stats["avg_scope_sums_per_example"], 1),
      "B =", round(stats["avg_fixed_sums_per_example"], 1))

In [ ]:
# Spot-check one saved scope example
from scopesum.data import load_examples
from scopesum.sum_insertion import Example

ds = load_examples(cfg.data.out_dir, "scope", "train")
ex = Example.from_dict(ds[0])
print(len(ds), "examples; first has", sum(k == 2 for k in ex.kind), "<sum> tokens,", sum(k == 1 for k in ex.kind), "raw tokens")
pieces = []
for tid, k in zip(ex.input_ids[:400], ex.kind[:400]):
    pieces.append(" ⟦SUM⟧ " if k == 2 else tok.decode([tid]))
print("".join(pieces))
!du -sh {cfg.data.out_dir}